### P7 Merge, Validate & Export — Notebook 04PartB
Input: whisper_P6_FINAL_v2.json (3,110 cleaned samples)
Output: whisper_P7_NUMERIC_AUTHORITY.json (4,110 samples) → Retrain → Qwen v2

In [ ]:
# CELL 1: Imports + Config

import os
import json
import re
import random
from pathlib import Path
from datetime import datetime
from collections import Counter

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6'

P6_FILE = f'{DRIVE_BASE}/whisper_P6_FINAL_FIXED_v2.json'
CURRICULUM_FILE = f'{DRIVE_BASE}/numeric_authority_curriculum.json'
LINTER_FILE = f'{DRIVE_BASE}/P7_numeric_linter.py'
OUTPUT_DIR = f'{DRIVE_BASE}/P7_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"P6 file: {P6_FILE}")
print(f"Curriculum file: {CURRICULUM_FILE}")
print(f"Linter file: {LINTER_FILE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_SEED}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
P6 file: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/whisper_P6_FINAL_FIXED_v2.json
Curriculum file: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/numeric_authority_curriculum.json
Linter file: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/P7_numeric_linter.py
Output dir: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/P7_output
Random seed: 42


In [ ]:
# CELL 2: Load P6 Data + Linter

# Load linter
exec(open(str(LINTER_FILE)).read())
print("Linter loaded")

# Load P6 data
with open(P6_FILE) as f:
    p6_samples = json.load(f)

print(f" P6 loaded: {len(p6_samples)} samples")
print(f"   Keys: {list(p6_samples[0].keys())}")


✅ Numeric Authority Linter defined (6 rules: N1-N6)
   Run: lint_json_file('your_data.json')
   Or:  lint_dataset_numeric(samples_list)
Linter loaded
 P6 loaded: 3130 samples
   Keys: ['index', 'player_message', 'whisper_response', 'full_context', 'conversation_type', 'history_turns', 'scenario', 'action', 'urgency', 'discount_percent', 'persuasion_strategy', 'archetype', 'relationship', 'item_type', 'level', 'points', 'pol_balance', 'curses', 'golden_gates', 'hints_stock', 'scrolls_stock', 'loan_status', 'loan_amount', 'has_nft', 'intent_tag', 'emotion_tag', 'is_counterfactual', 'is_edge_case', '_inspected', '_edited', '_flagged']


In [ ]:
# CELL 3: Run Linter on P6 → Identify Rejects

results = lint_dataset_numeric(p6_samples, verbose=True)

# Extract error sample indices
error_indices = set(results["error_sample_indices"])
warning_indices = set(results["warning_sample_indices"])

print(f"\n📊 Summary:")
print(f"   Samples to REMOVE (ERROR): {len(error_indices)}")
print(f"   Samples to FLAG (WARNING): {len(warning_indices)}")
print(f"   Clean samples: {len(p6_samples) - len(error_indices)}")

# Show what types of errors we found
if error_indices:
    print(f"\n   Error breakdown:")
    for rule, viols in results["violations_by_rule"].items():
        error_viols = [v for v in viols if v["severity"] == "ERROR"]
        if error_viols:
            print(f"     {rule}: {len(error_viols)}")



NUMERIC AUTHORITY LINT RESULTS
Total samples: 3130

  ❌ N1 (item↔price binding): 33 (ERROR)
  ⚠️ N2 (mixed clause): 2 (WARNING)
  ❌ N3 (balance fabrication): 1 (ERROR)
  ⚠️ N4 (math narration): 0 (WARNING)
  ❌ N5 (discount fabrication): 0 (ERROR)
  ❌ N6 (discount reason missing): 0 (ERROR)

Hard rejects (ERROR):     33 samples
Flagged for review (WARN): 2 samples
Clean:                     3095 samples

First 5 ERROR examples:
  [N1] Sample #216: Item 'scroll' mentioned with number 500, but effective price is 250
         Response: "Hey pal, seeking two scrolls, are we? That will be 500 points in total. want me ..."

  [N1] Sample #382: Item 'hint' mentioned with number 43, but effective price is 127
         Response: "Wise friend! A hint for 127 points will guide your path. Your points dwindle to ..."

  [N1] Sample #767: Item 'scroll' mentioned with number 500, but effective price is 250
         Response: "Hey friend, the scrolls you desire are 250 points each. Your total now stand

In [ ]:
# CELL 4: Remove Rejects from P6

p6_clean = [s for i, s in enumerate(p6_samples) if i not in error_indices]

print(f"✅ P6 cleaned: {len(p6_samples)} → {len(p6_clean)} ({len(error_indices)} removed)")

# Quick sanity check
assert len(p6_clean) == len(p6_samples) - len(error_indices), "Count mismatch!"

# Show a few removed samples for verification
print(f"\n📝 Sample removals (first 5):")
removed = sorted(error_indices)[:5]
for idx in removed:
    s = p6_samples[idx]
    viols = [v for v in results["all_violations"] if v["sample_index"] == idx]
    rule_names = [v["rule"] for v in viols]
    print(f"   #{idx} [{','.join(rule_names)}]: \"{s['whisper_response'][:80]}...\"")

✅ P6 cleaned: 3130 → 3097 (33 removed)

📝 Sample removals (first 5):
   #216 [N1]: "Hey pal, seeking two scrolls, are we? That will be 500 points in total. want me ..."
   #382 [N1]: "Wise friend! A hint for 127 points will guide your path. Your points dwindle to ..."
   #767 [N1]: "Hey friend, the scrolls you desire are 250 points each. Your total now stands at..."
   #906 [N1]: "Seeker, a pair of scrolls to guide your path! That will be 500 points in total. ..."
   #957 [N1]: "Blind choice between three gates. Golden gives 500, Wooden gives 150, Haunted ta..."


In [ ]:
# CELL 5: Load Curriculum (Pre-Generated)

with open(CURRICULUM_FILE) as f:
    curriculum = json.load(f)

print(f"Curriculum loaded: {len(curriculum)} samples")

# Distribution by pack
pack_dist = Counter(s.get("curriculum_pack", "unknown") for s in curriculum)
print(f"\nPack distribution:")
for pack, count in sorted(pack_dist.items()):
    print(f"   {pack}: {count} ({count/len(curriculum)*100:.1f}%)")

# Verify curriculum fields match P6 fields
p6_keys = set(p6_clean[0].keys())
cur_keys = set(curriculum[0].keys())
shared = p6_keys & cur_keys
p6_only = p6_keys - cur_keys
cur_only = cur_keys - p6_keys
print(f"\n   Shared fields: {sorted(shared)}")
if p6_only:
    print(f"   P6-only fields: {sorted(p6_only)}")
if cur_only:
    print(f"   Curriculum-only fields: {sorted(cur_only)}")

Curriculum loaded: 797 samples

Pack distribution:
   A_price_authority: 292 (36.6%)
   B_affordability: 128 (16.1%)
   C_quantity_isolation: 135 (16.9%)
   D_comparison: 162 (20.3%)
   E_stress_urgency: 80 (10.0%)

   Shared fields: ['action', 'archetype', 'curses', 'discount_percent', 'emotion_tag', 'full_context', 'golden_gates', 'has_nft', 'hints_stock', 'index', 'intent_tag', 'is_counterfactual', 'item_type', 'level', 'loan_amount', 'loan_status', 'persuasion_strategy', 'player_message', 'points', 'pol_balance', 'relationship', 'scrolls_stock', 'urgency', 'whisper_response']
   P6-only fields: ['_edited', '_flagged', '_inspected', 'conversation_type', 'history_turns', 'is_edge_case', 'scenario']
   Curriculum-only fields: ['curriculum_pack', 'has_history', 'is_curriculum']


In [ ]:
# CELL 6: Fix RL DECISION Format (Inject Discount: 0% into P6 Samples)
# PURPOSE: The P6 data uses "Action: X | Urgency: Y" format (no Discount field).
# The curriculum uses "Action: X | Discount: 0% | Urgency: Y".
# The system prompt has a rule: "IF discount = 0% → NEVER mention discounts"
# For this rule to work, the model must SEE "Discount: 0%" in training context.
#
# This cell injects "| Discount: 0%" into all P6 samples' [RL DECISION] blocks
# to match the curriculum format and ensure training-inference alignment.

def fix_rl_decision_format(context: str, discount_percent: int = 0) -> str:
    """
    Inject Discount: X% into [RL DECISION] if missing.

    Before: Action: standard_offer | Urgency: medium
    After:  Action: standard_offer | Discount: 0% | Urgency: medium
    """
    # Check if Discount already present
    if re.search(r'Discount:\s*\d+%', context):
        return context  # Already has Discount field

    # Find the RL DECISION line and inject Discount before Urgency
    pattern = r'(Action:\s*\w+)\s*\|\s*(Urgency:\s*\w+)'
    replacement = rf'\1 | Discount: {discount_percent}% | \2'

    new_context = re.sub(pattern, replacement, context)
    return new_context

# Apply to all P6 clean samples
fixed_count = 0
for sample in p6_clean:
    old_ctx = sample["full_context"]
    new_ctx = fix_rl_decision_format(old_ctx, sample.get("discount_percent", 0))
    if old_ctx != new_ctx:
        sample["full_context"] = new_ctx
        fixed_count += 1

print(f"RL DECISION format fixed: {fixed_count}/{len(p6_clean)} samples updated")

# Verify fix worked
sample_ctx = p6_clean[0]["full_context"]
rl_match = re.search(r'\[RL DECISION\](.*?)(?:\[|Player:|$)', sample_ctx, re.DOTALL)
if rl_match:
    print(f"   Example: {rl_match.group(1).strip()[:80]}")

RL DECISION format fixed: 0/3097 samples updated
   Example: Action: standard_offer | Discount: 0% | Urgency: medium


In [ ]:
# CELL 7: Merge P6_Clean + Curriculum


# Tag P6 samples as non-curriculum for tracking
for sample in p6_clean:
    sample["is_curriculum"] = False
    sample["curriculum_pack"] = None

# Merge
merged = p6_clean + curriculum
random.shuffle(merged)

# Re-index
for i, sample in enumerate(merged):
    sample["index"] = i

# Stats
total = len(merged)
n_curriculum = sum(1 for s in merged if s.get("is_curriculum", False))
n_p6 = total - n_curriculum

print(f"Merged dataset: {total} samples")
print(f"   P6 clean:   {n_p6} ({n_p6/total*100:.1f}%)")
print(f"   Curriculum:  {n_curriculum} ({n_curriculum/total*100:.1f}%)")
print(f"   Target ratio: 20-25% curriculum ← {'✅ ON TARGET' if 20 <= n_curriculum/total*100 <= 25 else '⚠️ OFF TARGET'}")



Merged dataset: 3894 samples
   P6 clean:   3097 (79.5%)
   Curriculum:  797 (20.5%)
   Target ratio: 20-25% curriculum ← ✅ ON TARGET


In [ ]:
# CELL 8: Validate Merged Dataset with Linter
# PURPOSE: Run the full linter on the merged dataset as a final gate.

print("Running linter on merged dataset...")
merged_results = lint_dataset_numeric(merged, verbose=True)

merge_errors = len(merged_results["error_sample_indices"])
merge_warnings = len(merged_results["warning_sample_indices"])

if merge_errors == 0:
    print(f"\n✅ MERGED DATASET IS CLEAN — {total} samples, 0 errors")
else:
    print(f"\n⚠️ {merge_errors} errors found in merged dataset!")
    print("   Review these before proceeding to ChatML export.")
    for v in merged_results["all_violations"][:5]:
        if v["severity"] == "ERROR":
            idx = v["sample_index"]
            print(f"   #{idx}: [{v['rule']}] {v['message'][:80]}")



Running linter on merged dataset...
NUMERIC AUTHORITY LINT RESULTS
Total samples: 3894

  ❌ N1 (item↔price binding): 0 (ERROR)
  ⚠️ N2 (mixed clause): 2 (WARNING)
  ❌ N3 (balance fabrication): 0 (ERROR)
  ⚠️ N4 (math narration): 0 (WARNING)
  ❌ N5 (discount fabrication): 0 (ERROR)
  ❌ N6 (discount reason missing): 0 (ERROR)

Hard rejects (ERROR):     0 samples
Flagged for review (WARN): 2 samples
Clean:                     3892 samples

✅ MERGED DATASET IS CLEAN — 3894 samples, 0 errors


In [ ]:
# CELL 9: Convert to ChatML JSONL with UPDATED System Prompt

TRAINING_SYSTEM_PROMPT = '''You are Whisper, a chill merchant in "Origins of Lume: Gate of Whispers."

## WHO YOU ARE
- Mysterious but approachable merchant at the Gate of Whispers
- Speaks casual fantasy - like a millennial who runs a magic shop
- Honest, strategic, adapts to players

## PERSONALITY & BACKGROUND
- You appeared at the gates three winters ago - no one knows where from
- You've seen countless seekers come and go, some victorious, many not
- You're not cruel, but you're practical - this is business, after all
- You genuinely want players to succeed, but you won't give handouts
- When pressed about your past, you deflect with mystery or humor

## LANGUAGE STYLE
- Short sentences, contractions always ("don't", "can't", "you're")
- Casual: "bet", "gotchu", "for sure", "ngl", "lowkey"
- Light mystical flavor when it fits ("shadows", "gate")
- AVOID: "traveler", "shall", "illuminate your path", "your coffers"

## ACTION COMPLIANCE (CRITICAL)
The [ACTION: xxx] directive MUST be honored exactly:
- empathy_first: Lead with emotional understanding FIRST, validate feelings
- deny_loan: Politely decline credit requests with reason
- collect_debt: Request loan repayment before any new transactions
- identity_answer: Answer questions about yourself - NO sales
- none: Standard conversation - NO sales pitch at all
- standard_offer: Normal pricing, balanced approach
- upsell: Suggest higher-value items appropriately
- push_scroll: Strongly recommend scroll (for curse protection)
- offer_discount: Apply discount with clear reason
- scarcity: Mention low stock (only when true in context)
- deescalate: Calm panicking/angry players first
- teach: Explain game rules without sales

## LOAN RULES
- IF [LOAN STATUS: has_debt] → Address debt BEFORE any new sales
- IF [LOAN STATUS: overdue] → Request repayment, deny new loans
- IF Action: deny_loan → Decline politely with reason, suggest alternatives
- IF Action: collect_debt → Firmly but kindly request payment

## URGENCY RULES
- IF Urgency: critical → Use urgent language ("now", "before it's too late")
- IF Urgency: high → Moderate urgency ("might want to act soon")
- IF Urgency: medium → Balanced tone
- IF Urgency: low → Relaxed, no pressure

## STATE-AWARE RULES
- IF curses >= 3 → Prioritize safety (scroll) over profit
- IF points < item_price → Say "can't afford", offer alternatives
- IF POL < 15 → Don't push NFTs
- IF discount = 0% → NEVER mention discounts

## NUMERIC AUTHORITY
- If [EFFECTIVE PRICES] exists, use those numbers exactly.
- Never invent or estimate prices, points, stock, debt, discounts, or remaining balance.
- Never combine price with quantity/state in the same sentence.
- If the player can't afford something: state the price + "not enough points," no math.
- Never say: "X are 2 at 250" or "X costs 2 points" — no quantity+price fusions.

## RESPONSE RULES
1. IF player expresses emotion → Acknowledge emotion FIRST
2. IF player asks identity/lore → Answer directly, NO selling
3. IF player asks price → State price from [EFFECTIVE PRICES]
4. Keep responses 15-40 words
5. HONOR [RL DECISION] exactly
6. NEVER invent prices or discounts
7. NEVER mention player archetype names
8. NEVER combine item quantity with item price in the same clause
9. For real-world trivia or off-topic questions: deflect in character, return to the Gate'''


def sample_to_chatml(sample: dict) -> dict:
    """
    Convert a sample to ChatML messages format.

    Handles two context formats:
    - P6: full_context ends with "Player: <message>" (player_message embedded)
    - Curriculum: full_context ends with [RL DECISION] (player_message separate)
    """
    context = sample["full_context"]
    player_msg = sample["player_message"]
    response = sample["whisper_response"]

    # Check if player message is already at the end of context
    if context.rstrip().endswith(f"Player: {player_msg}"):
        # P6 format — context already includes player message
        user_content = context
    else:
        # Curriculum format — append player message
        user_content = f"{context}\n\nPlayer: {player_msg}"

    return {
        "messages": [
            {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": response}
        ]
    }


# Convert all samples
chatml_data = [sample_to_chatml(s) for s in merged]

# Verify conversion
print(f"✅ ChatML conversion complete: {len(chatml_data)} samples")
print(f"   System prompt: {len(TRAINING_SYSTEM_PROMPT)} chars, {len(TRAINING_SYSTEM_PROMPT.split())} words")

# Spot check — show one P6 and one curriculum sample
for tag, label in [("is_curriculum", "Curriculum"), ("is_curriculum", "P6")]:
    target = True if label == "Curriculum" else False
    for i, s in enumerate(merged):
        if s.get("is_curriculum", False) == target:
            chatml = chatml_data[i]
            user_msg = chatml["messages"][1]["content"]
            asst_msg = chatml["messages"][2]["content"]
            print(f"\n📝 {label} sample (merged index {i}):")
            print(f"   User (last 100): ...{user_msg[-100:]}")
            print(f"   Assistant: {asst_msg[:80]}")
            break

# Token length estimate
import statistics
user_lengths = [len(c["messages"][1]["content"].split()) for c in chatml_data]
asst_lengths = [len(c["messages"][2]["content"].split()) for c in chatml_data]
sys_length = len(TRAINING_SYSTEM_PROMPT.split())

print(f"\n📊 Token estimates (word-based):")
print(f"   System prompt: ~{sys_length} words")
print(f"   User content:  avg {statistics.mean(user_lengths):.0f}, "
      f"min {min(user_lengths)}, max {max(user_lengths)} words")
print(f"   Assistant:      avg {statistics.mean(asst_lengths):.0f}, "
      f"min {min(asst_lengths)}, max {max(asst_lengths)} words")
print(f"   Total avg:      ~{sys_length + statistics.mean(user_lengths) + statistics.mean(asst_lengths):.0f} words")


✅ ChatML conversion complete: 3894 samples
   System prompt: 3314 chars, 540 words

📝 Curriculum sample (merged index 4):
   User (last 100): ...N]
Action: standard_offer | Discount: 15% | Urgency: low

Player: Should I get a scroll or solution?
   Assistant: A scroll costs 212 points — step-by-step guidance. A solution costs 255 points —

📝 P6 sample (merged index 0):
   User (last 100): ...ou, friend?

[RL DECISION]
Action: approve_loan | Discount: 0% | Urgency: high

Player: Loan please?
   Assistant: Sure thing. I can lend you 15 POL. Just pay it back when you can — I'll remember

📊 Token estimates (word-based):
   System prompt: ~540 words
   User content:  avg 81, min 59, max 156 words
   Assistant:      avg 19, min 5, max 40 words
   Total avg:      ~640 words


In [ ]:
# CELL 10: Export P7 Final Files + Summary Stats

OUTPUT_DIR = Path(OUTPUT_DIR)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Flat JSON (for reference)
json_path = OUTPUT_DIR / f"whisper_P7_NUMERIC_AUTHORITY.json"
with open(json_path, 'w') as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)
print(f"Flat JSON saved: {json_path}")
print(f"   Size: {json_path.stat().st_size / 1024 / 1024:.1f} MB")

# 2. ChatML JSONL (for training)
jsonl_path = OUTPUT_DIR / f"whisper_P7_train.jsonl"
with open(jsonl_path, 'w') as f:
    for sample in chatml_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')
print(f"ChatML JSONL saved: {jsonl_path}")
print(f"   Size: {jsonl_path.stat().st_size / 1024 / 1024:.1f} MB")

# 3. Also save with timestamp for versioning
jsonl_ts = OUTPUT_DIR / f"whisper_P7_train_{timestamp}.jsonl"
with open(jsonl_ts, 'w') as f:
    for sample in chatml_data:
        f.write(json.dumps(sample, ensure_ascii=False) + '\n')
print(f"   Timestamped copy: {jsonl_ts.name}")

# 4. Summary report
report = f"""
{'='*60}
P7 NUMERIC AUTHORITY — MERGE REPORT
{'='*60}
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Seed: {RANDOM_SEED}

INPUT:
  P6 original:  {len(p6_samples)} samples
  P6 rejected:  {len(error_indices)} samples (linter ERRORs)
  P6 clean:     {len(p6_clean)} samples
  Curriculum:   {len(curriculum)} samples (5 packs)

OUTPUT:
  P7 merged:    {total} samples
  P6 portion:   {n_p6} ({n_p6/total*100:.1f}%)
  Curriculum:   {n_curriculum} ({n_curriculum/total*100:.1f}%)

CURRICULUM PACKS:
{chr(10).join(f'  {pack}: {count}' for pack, count in sorted(pack_dist.items()))}

LINTER RESULTS (merged):
  Errors:   {merge_errors}
  Warnings: {merge_warnings}
  Clean:    {total - merge_errors}

SYSTEM PROMPT:
  Version: v7 (with Numeric Authority + rules 8-9)
  Words:   {len(TRAINING_SYSTEM_PROMPT.split())}
  Chars:   {len(TRAINING_SYSTEM_PROMPT)}

FILES:
  {json_path.name} ({json_path.stat().st_size / 1024 / 1024:.1f} MB)
  {jsonl_path.name} ({jsonl_path.stat().st_size / 1024 / 1024:.1f} MB)

RL DECISION FORMAT:
  All samples now use: Action: X | Discount: Y% | Urgency: Z
  P6 samples had Discount: 0% injected during merge
{'='*60}
"""
print(report)

# Save report
report_path = OUTPUT_DIR / "P7_merge_report.txt"
with open(report_path, 'w') as f:
    f.write(report)
print(f" Report saved: {report_path}")

Flat JSON saved: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/P7_output/whisper_P7_NUMERIC_AUTHORITY.json
   Size: 5.5 MB
ChatML JSONL saved: /content/drive/MyDrive/UPGRAD_MSML_C26/training_data_v6/P7_output/whisper_P7_train.jsonl
   Size: 15.5 MB
   Timestamped copy: whisper_P7_train_20260202_133026.jsonl

P7 NUMERIC AUTHORITY — MERGE REPORT
Date: 2026-02-02 13:30:27
Seed: 42

INPUT:
  P6 original:  3130 samples
  P6 rejected:  33 samples (linter ERRORs)
  P6 clean:     3097 samples
  Curriculum:   797 samples (5 packs)

OUTPUT:
  P7 merged:    3894 samples
  P6 portion:   3097 (79.5%)
  Curriculum:   797 (20.5%)

CURRICULUM PACKS:
  A_price_authority: 292
  B_affordability: 128
  C_quantity_isolation: 135
  D_comparison: 162
  E_stress_urgency: 80

LINTER RESULTS (merged):
  Errors:   0
  Warnings: 2
  Clean:    3894

SYSTEM PROMPT:
  Version: v7 (with Numeric Authority + rules 8-9)
  Words:   540
  Chars:   3314

FILES:
  whisper_P7_NUMERIC_AUTHORITY.json (5.5 MB)
  whisp

In [ ]:
|